In [1]:
# [CELL A1] STUDENT CONFIGURATION (same as main activity)
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-1227"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
RANDOM_SEED = int(id_digits) % (2**32)
N_SELECT = 5 + (SEED_DIGIT % 3)
ROBUST_LIMIT = 3.5

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("Random Seed:", RANDOM_SEED)
print("Features to Select:", N_SELECT)

Student: Solis TUPM-26-1227
Seed Digit: 7
Random Seed: 261227
Features to Select: 6


In [2]:
# [CELL A2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
output_path = OUTPUT_DIR / "measurement_quality.csv"

for path in [measurement_path, label_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("Dataset files verified.")

Workspace: C:\Users\n0n3\Lab7_NumPy_Pandas_Solis\Lab7_NumPy_Pandas_Additional_Exercises
Dataset files verified.


In [3]:
# [CELL A3] LOAD DATA AND IDENTIFY MISSING / CONSTANT FEATURES
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)
assert raw_measurements.shape[0] == len(labels)

n_features = raw_measurements.shape[1]
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0
has_missing = missing.any(axis=0)

observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)

usable_indices = np.flatnonzero(~(all_missing | constant))

print("Shape:", raw_measurements.shape)
print("Features With Missing Values:", has_missing.sum())
print("Entirely Missing Features:", all_missing.sum())
print("Constant Features:", constant.sum())
print("Usable Features:", len(usable_indices))

Shape: (1567, 590)
Features With Missing Values: 538
Entirely Missing Features: 0
Constant Features: 116
Usable Features: 474


In [4]:
# [CELL A4] REPRODUCIBLE FEATURE SELECTION
rng = np.random.default_rng(RANDOM_SEED)

selected_indices = np.sort(
    rng.choice(usable_indices, size=N_SELECT, replace=False)
)
selected_names = feature_names[selected_indices]

selected = raw_measurements[:, selected_indices]   # rows (and labels) stay aligned

print("Selected Features:", selected_names.tolist())
print("Selected Block Shape:", selected.shape)

Selected Features: ['Feature_019', 'Feature_020', 'Feature_050', 'Feature_076', 'Feature_203', 'Feature_589']
Selected Block Shape: (1567, 6)


In [5]:
# [CELL A5] FEATURE-WISE STATISTICS (NumPy, NaN-aware)
sel_missing = np.isnan(selected)
sel_observed = (~sel_missing).sum(axis=0)

means = np.nanmean(selected, axis=0)
medians = np.nanmedian(selected, axis=0)
sds = np.nanstd(selected, axis=0, ddof=1)
missing_pct = 100 * sel_missing.mean(axis=0)

print("Means:", means)
print("Medians:", medians)
print("Sample SDs:", sds)
print("Missing %:", missing_pct)

Means: [ 12.481    1.4051 632.2542  -0.0294  30.0731  99.6701]
Medians: [ 12.4996   1.406  631.3709  -0.0289  30.097   71.9005]
Sample SDs: [ 0.218   0.0167  8.644   0.0332 17.4618 93.8919]
Missing %: [0.6382 0.     0.0638 1.5316 0.3829 0.0638]


In [6]:
# [CELL A6] ROBUST STANDARDIZATION (MEDIAN / MAD)
mad = np.nanmedian(np.abs(selected - medians), axis=0)
zero_mad = mad == 0

# Modified z-score: 0.6745 * (x - median) / MAD. Zero-MAD features are
# set to 0 (never flagged) instead of dividing by zero.
robust_z = np.divide(
    0.6745 * (selected - medians), mad,
    out=np.zeros_like(selected), where=~zero_mad
)

print("MAD:", mad)
print("Zero-MAD Features:", int(zero_mad.sum()))
print("Robust Z Shape:", robust_z.shape)

MAD: [ 0.0435  0.0092  6.1545  0.0226  4.068  32.3477]
Zero-MAD Features: 0
Robust Z Shape: (1567, 6)


In [7]:
# [CELL A7] FLAG UNUSUAL OBSERVATIONS (BOOLEAN INDEXING)
flags = np.abs(robust_z) > ROBUST_LIMIT      # NaN compares False -> never flagged

flag_counts = flags.sum(axis=0)
flag_pct = 100 * flag_counts / sel_observed
total_flagged = int(flags.sum())

flagged_values = selected[flags]             # the actual flagged measurements
rows_with_flag = np.flatnonzero(flags.any(axis=1))

print("Flagged per Feature:", flag_counts)
print("Flagged %:", flag_pct)
print("Total Flagged Observations:", total_flagged, "| Records Affected:", len(rows_with_flag))
print("Flagged Records With Fail Label:", int((labels[rows_with_flag] == 1).sum()))

Flagged per Feature: [ 24  14   1   2  12 110]
Flagged %: [1.5414 0.8934 0.0639 0.1296 0.7687 7.0243]
Total Flagged Observations: 163 | Records Affected: 158
Flagged Records With Fail Label: 8


In [8]:
# [CELL A8] RANK FEATURES AND EXPORT CSV
summary = pd.DataFrame({
    "Feature": selected_names,
    "Mean": means,
    "Median": medians,
    "Sample_SD": sds,
    "MAD": mad,
    "Missing_Pct": missing_pct,
    "Flagged_Count": flag_counts,
    "Flagged_Pct": flag_pct,
}).sort_values(["Flagged_Pct", "Flagged_Count"], ascending=False)

summary.insert(0, "Rank", np.arange(1, len(summary) + 1))
summary = summary.reset_index(drop=True)
summary.to_csv(output_path, index=False)

print(summary.round(4).to_string(index=False))

 Rank     Feature     Mean   Median  Sample_SD     MAD  Missing_Pct  Flagged_Count  Flagged_Pct
    1 Feature_589  99.6701  71.9005    93.8919 32.3477       0.0638            110       7.0243
    2 Feature_019  12.4810  12.4996     0.2180  0.0435       0.6382             24       1.5414
    3 Feature_020   1.4051   1.4060     0.0167  0.0092       0.0000             14       0.8934
    4 Feature_203  30.0731  30.0970    17.4618  4.0680       0.3829             12       0.7687
    5 Feature_076  -0.0294  -0.0289     0.0332  0.0226       1.5316              2       0.1296
    6 Feature_050 632.2542 631.3709     8.6440  6.1545       0.0638              1       0.0639


In [9]:
# [CELL A9] VERIFY VECTORIZED RESULTS WITH PANDAS AND CSV RELOAD
check = pd.DataFrame(selected, columns=selected_names)

pd_median = check.median()
pd_mad = (check - pd_median).abs().median()
pd_z = 0.6745 * (check - pd_median) / pd_mad.where(pd_mad > 0)
pd_flags = pd_z.abs().gt(ROBUST_LIMIT).fillna(False)

stats_ok = (
    np.allclose(means, check.mean().to_numpy())
    and np.allclose(sds, check.std(ddof=1).to_numpy())
    and np.allclose(medians, pd_median.to_numpy())
    and np.allclose(missing_pct, 100 * check.isna().mean().to_numpy())
)
flags_ok = np.array_equal(flags, pd_flags.to_numpy())
rows_ok = np.array_equal(
    selected, raw_measurements[:, selected_indices], equal_nan=True
) and len(labels) == selected.shape[0]

loaded = pd.read_csv(output_path)
csv_ok = (
    loaded["Flagged_Count"].sum() == total_flagged
    and list(loaded["Feature"]) == list(summary["Feature"])
)

verified = bool(stats_ok and flags_ok and rows_ok and csv_ok)

print("Statistics Match Pandas:", stats_ok)
print("Flags Match Pandas:", flags_ok)
print("Row/Label Alignment:", rows_ok)
print("CSV Reload Verified:", csv_ok)

Statistics Match Pandas: True
Flags Match Pandas: True
Row/Label Alignment: True
CSV Reload Verified: True


In [10]:
# [CELL A10] ASSESSMENT DATA (one line per answer)
top = summary.iloc[0]
top_idx = int(top["Feature"].split("_")[1])

print("Selected Features:", ", ".join(str(i) for i in selected_indices))
print("Highest Missing Percentage:", f"{missing_pct.max():.2f}%")
print("Feature with Highest Flagged Percentage:",
    f"{top_idx} ({top['Flagged_Pct']:.2f}%)")
print("Total Flagged Observations:", total_flagged)
print("Vectorized Results Verified:", "Yes" if verified else "No")

Selected Features: 19, 20, 50, 76, 203, 589
Highest Missing Percentage: 1.53%
Feature with Highest Flagged Percentage: 589 (7.02%)
Total Flagged Observations: 163
Vectorized Results Verified: Yes
